In [1]:
import os
print(os.getcwd())

C:\Users\krith\Documents\skin-disease-detection-ai\notebooks


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [3]:
METADATA_PATH = "../dataset/HAM10000_metadata.csv"
IMAGE_DIR_1 = "../dataset/all_images/HAM10000_images_part_1"
IMAGE_DIR_2 = "../dataset/all_images/HAM10000_images_part_2"
OUTPUT_DIR = "../processed_data"

IMG_SIZE = 224

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
df = pd.read_csv(METADATA_PATH)
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  object 
 1   image_id      10015 non-null  object 
 2   dx            10015 non-null  object 
 3   dx_type       10015 non-null  object 
 4   age           9958 non-null   float64
 5   sex           10015 non-null  object 
 6   localization  10015 non-null  object 
dtypes: float64(1), object(6)
memory usage: 547.8+ KB


In [6]:
df.isnull().sum()

lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64

In [7]:
image_path_map = {}

for folder in [IMAGE_DIR_1, IMAGE_DIR_2]:
    for img in os.listdir(folder):
        img_id = img.replace(".jpg", "")
        image_path_map[img_id] = os.path.join(folder, img)

df["image_path"] = df["image_id"].map(image_path_map)
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,../dataset/all_images/HAM10000_images_part_1\I...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,../dataset/all_images/HAM10000_images_part_1\I...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,../dataset/all_images/HAM10000_images_part_1\I...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,../dataset/all_images/HAM10000_images_part_1\I...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,../dataset/all_images/HAM10000_images_part_2\I...


In [8]:
before = len(df)
df = df[df["image_path"].notnull()].reset_index(drop=True)
after = len(df)

print("Before cleaning:", before)
print("After cleaning :", after)

Before cleaning: 10015
After cleaning : 10015


In [9]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["dx"])

df[["dx", "label"]].drop_duplicates()

,dx,label
0,bkl,2
64,nv,5
1095,df,3
1211,mel,4
2320,vasc,6
2462,bcc,1
9687,akiec,0


In [10]:
IMG_SIZE = 64   # 🔥 reduced size

images = []
labels = []

print("Preprocessing images...")

for _, row in tqdm(df.iterrows(), total=len(df)):
    img = cv2.imread(row["image_path"])
    
    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0

    images.append(img.flatten())
    labels.append(row["label"])

Preprocessing images...


100%|████████████████████████████████████████████████████████████████████████████| 10015/10015 [02:52<00:00, 57.98it/s]


In [11]:
X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

final_df = pd.DataFrame(X)
final_df["label"] = y

final_df.head()

,0,1,2,3,4,5,6,7,8,9,...,12279,12280,12281,12282,12283,12284,12285,12286,12287,label
0,0.756863,0.611765,0.772549,0.737255,0.596078,0.760784,0.760784,0.568627,0.745098,0.772549,...,0.725490,0.596078,0.709804,0.737255,0.611765,0.725490,0.713726,0.588235,0.678431,2
1,0.082353,0.047059,0.098039,0.090196,0.054902,0.105882,0.141176,0.098039,0.164706,0.243137,...,0.101961,0.058824,0.113725,0.105882,0.058824,0.109804,0.094118,0.054902,0.101961,2
2,0.725490,0.505882,0.541176,0.749020,0.525490,0.580392,0.780392,0.576471,0.627451,0.772549,...,0.603922,0.470588,0.533333,0.580392,0.454902,0.498039,0.447059,0.309804,0.317647,2
3,0.090196,0.039216,0.066667,0.133333,0.078431,0.117647,0.247059,0.137255,0.184314,0.356863,...,0.101961,0.054902,0.070588,0.101961,0.039216,0.070588,0.090196,0.031373,0.050980,2
4,0.545098,0.368627,0.458824,0.619608,0.447059,0.545098,0.701961,0.525490,0.639216,0.749020,...,0.545098,0.415686,0.458824,0.431373,0.305882,0.364706,0.262745,0.156863,0.215686,2


In [12]:
train_df, test_df = train_test_split(
    final_df,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (8012, 12289)
Test shape : (2003, 12289)


In [13]:
train_df.to_csv("../processed_data/train.csv", index=False)
test_df.to_csv("../processed_data/test.csv", index=False)

print("✅ Preprocessing completed successfully")

✅ Preprocessing completed successfully
